# 10c -- CellRank2 Fate-Mapping 拟时序 (独立 Notebook)

本 notebook 对上皮谱系做 **CellRank2 完整 fate-mapping 分析**，是拟时序方法系列的第二个独立 notebook。
与 `D04_pseudotime.ipynb`（综合：熵 + CytoTRACE + root + Monocle3）和 `D05_pseudotime_monocle3.ipynb`
（Monocle3 独立）并行存在，PI 可分别跑、分别看、最后横向比较不同方法的 pseudotime 结果。

## 方法原理

**CellRank2** 是单细胞 fate-mapping 框架，核心工作流：
1. **Markov 转移矩阵**：从 CytoTRACE 分化潜能评分或 pseudotime 方向构建细胞间转移概率矩阵
2. **GPCCA（Generalized Perron Cluster Cluster Analysis）**：基于转移矩阵的 Schur 分解，识别
   粗粒化"宏观状态"（macrostates），自动推断谱系终末状态
3. **命运概率（Fate Probabilities）**：计算每个细胞到达各终末状态的概率——这是 CellRank2
   区别于其他拟时序工具的看家本领

相比 D04_pseudotime.ipynb 中仅用 `CytoTRACEKernel` 算单个潜能分数，
本 notebook 跑通 CellRank2 的完整 fate-mapping 管线：kernel 构建 → 转移矩阵 → GPCCA 分解
→ 终末状态识别 → 命运概率。

## 输入与输出

| 项目 | 路径/字段 |
|------|-----------|
| 上游输入 | `UPSTREAM_PATH`（默认 `results/06_annotated_v1.h5ad`）|
| Root 来源 | 优先读取上游 `adata.uns["root_cluster"]`；否则干细胞 marker fallback |
| 输出 h5ad | `OUTPUT_PATH`（默认 `results/10c_pseudotime_cellrank2_v1.h5ad`）|
| 新增 obs 列 | `cellrank2_cytotrace`、`cellrank2_macrostate`、`cellrank2_terminal_state` |
| 新增 obsm key | `cellrank2_fate_probabilities` |
| Figure | `results/figures/10c_macrostates_umap.png`、`10c_fate_probabilities_umap.png` |

## 与其他拟时序 notebook 的关系

| Notebook | 方法 | 产物 obs 列 |
|----------|------|-------------|
| `D04_pseudotime.ipynb` | 综合（熵 + CytoTRACE + Monocle3）| `entropy` / `cytotrace_score` / `pseudotime_monocle3_v1` |
| `D05_pseudotime_monocle3.ipynb` | Monocle3 独立 | `pseudotime_monocle3_v1` |
| `D06_pseudotime_cellrank2.ipynb`（本 notebook）| CellRank2 | `cellrank2_cytotrace` / `cellrank2_macrostate` / `cellrank2_terminal_state` |

PI 可以在 Jupyter 中打开各 notebook 产出的 h5ad，交叉比较不同方法的 pseudotime/fate 结果是否一致。


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（细胞注释），读 `06_annotated_v*.h5ad`
- **同级**：`D04_pseudotime.ipynb`（综合拟时序）、`D05_pseudotime_monocle3.ipynb`（Monocle3 独立）
- **下游**：本 stage 产出 checkpoint 供横向比较或后续可视化使用

### 为什么要迭代回跑？
CellRank2 fate-mapping 的质量取决于几个关键参数：
- **kernel 选择**（`CELLRANK_KERNEL`）：CytoTRACE vs Pseudotime 决定转移矩阵的方向性
- **macrostates 数量**（`CELLRANK_N_STATES`）：太少合并异质群体，太多过拟合噪音
- **邻居数**（`CELLRANK_N_NEIGHBORS`）：影响转移矩阵的局部平滑程度
- **root cluster 选择**：影响 CytoTRACEKernel 的分化方向判断

### 如何回跑（三步操作）
1. 改 `UPSTREAM_PATH`——指向要复用的上游文件版本
2. 改 `OUTPUT_PATH`——bump 版本号 `_v1` → `_v2`
3. 调整参数（在下方 `# === PARAMS ===` 区域）→ 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。旧版 `.h5ad` 文件不覆盖不删除
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）
- **`promoted`**：PI 审查后确认可传给下游使用的正式版本
- **下游取数**：后续 stage 的 `UPSTREAM_PATH` 指向你决定采用的版本即可

### 追溯链（自动写入 h5ad 的 `adata.uns`）
- `stage` = `"10c_pseudotime_cellrank2"`
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号
- `10c_pseudotime_cellrank2_v1` = 方法参数嵌套 dict


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH           -- 06 注释结果 h5ad
# OUTPUT_PATH             -- 本 notebook 产出 checkpoint
# RUN_CELLRANK2           -- 布尔开关：PI 可关闭整段 CellRank2
# CELLRANK_KERNEL         -- 主核选择："cytotrace"（默认，无需依赖其他方法）
#                            或 "pseudotime"（需上游有 pseudotime 列）
# CELLRANK_N_NEIGHBORS    -- 邻居图邻居数（未建时使用）
# CELLRANK_N_STATES       -- GPCCA macrostates 数量，None=自动（eigengap 推断）
# PSEUDOTIME_KEY          -- KERNEL="pseudotime" 时用哪列；None 自动探测
# USE_UPSTREAM_ROOT       -- 优先使用上游 adata.uns["root_cluster"]
# CLUSTER_KEY             -- 用于按簇聚合的 obs 列（root 识别 fallback）
# CELL_TYPE_COL           -- 优先使用的细胞类型列
# EPITHELIAL_CLUSTERS     -- 上皮谱系 cluster ID 列表；None=全体细胞
# AUTO_SUBSET_EPITHELIAL  -- 自动检测并筛选上皮细胞做轨迹分析
# EPITHELIAL_LABEL_COL    -- 从哪个列检测上皮标签
# EPITHELIAL_KEYWORDS     -- 上皮关键词列表（小写匹配）
# STEM_MARKERS            -- 已知干细胞/祖细胞 marker 基因列表（root fallback 时使用）

# === 版本变量（单点定义，回跑时只改这里，下游全部引用）===
# 为什么用字符串而非整数：uns["version"] 历来存 "v1"，直接引用变量即保持语义不变；
# 回跑时把 "v1" 改成 "v2" 即可，旧结果不覆盖。
UPSTREAM_VERSION = "v1"   # 上游输入 checkpoint 的版本号（本 stage 读取的 06/D04 产物）
OUTPUT_VERSION   = "v1"   # 本 stage 产出 checkpoint 的版本号

UPSTREAM_PATH = f"results/06_annotated_{UPSTREAM_VERSION}.h5ad"
OUTPUT_PATH   = f"results/10c_pseudotime_cellrank2_{OUTPUT_VERSION}.h5ad"

# === CellRank2 开关 ===
RUN_CELLRANK2 = True  # PI 可设为 False 跳过整段 CellRank2

# === CellRank2 方法参数 ===
CELLRANK_KERNEL = "cytotrace"  # 核选择："cytotrace"（默认）| "pseudotime"
CELLRANK_N_NEIGHBORS = 30     # 邻居图邻居数（未建时使用）
CELLRANK_N_STATES = None       # GPCCA macrostates 数量；None=自动（eigengap 推断）
PSEUDOTIME_KEY = None          # KERNEL="pseudotime" 时指定 pseudotime 列；
                               # None 则自动探测：pseudotime_monocle3_v1 > dpt_pseudotime > pseudotime

# === Root 识别 ===
USE_UPSTREAM_ROOT = True  # 优先使用上游 adata.uns["root_cluster"]；False=强制 stem marker fallback

CLUSTER_KEY   = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

# === 上皮筛选 ===
EPITHELIAL_CLUSTERS = None  # None=全体细胞；PI 按需设如 ["0", "3", "5"]

AUTO_SUBSET_EPITHELIAL = True
EPITHELIAL_LABEL_COL = "cell_type_final_v1"
EPITHELIAL_KEYWORDS = ["epithelial", "parietal", "chief", "mucous", "foveolar",
                       "pit", "neck", "spem", "im_", "intestinal", "goblet"]

# === Root fallback：干细胞 marker（仅在上游无 root_cluster 时使用）===
STEM_MARKERS = [
    "LGR5", "SOX9", "MKI67", "OLFM4", "TERT",
    "AXIN2", "LRIG1", "TFF2", "MUC6", "MUC5AC",
]
# 以上为胃上皮干细胞/祖细胞常用 marker。基因不存在时自动跳过。


In [ ]:
# === setup：sys.path + env_check + 导入 + 切换目录 ===
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc

_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

# 环境自检（与其他 notebook 一致的 env_check 模式）
try:
    from scrna_integration.platform import env_check
    env_check(expected_env="scrna-integration")
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cellrank as cr
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  cellrank {cr.__version__}  |  numpy {np.__version__}")


In [ ]:
# === 加载上游 h5ad ===
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")

# 确定实际使用的分组列
if CELL_TYPE_COL in adata.obs.columns:
    _group_col = CELL_TYPE_COL
    print(f"使用细胞类型列: {CELL_TYPE_COL}")
    _ct = adata.obs[CELL_TYPE_COL].dropna().astype(str)
    print(f"  细胞类型: {sorted(_ct.unique())}")
elif CLUSTER_KEY in adata.obs.columns:
    _group_col = CLUSTER_KEY
    print(f"CELL_TYPE_COL 不存在，fallback 到 CLUSTER_KEY: {CLUSTER_KEY}")
    print(f"  簇数: {adata.obs[CLUSTER_KEY].nunique()}")
else:
    raise KeyError(
        f"CELL_TYPE_COL '{CELL_TYPE_COL}' 和 "
        f"CLUSTER_KEY '{CLUSTER_KEY}' 都不在 obs 列中"
    )

# 检查 embedding
_has_umap = "X_umap" in adata.obsm
print(f"X_umap: {_has_umap}  |  obsm keys: {list(adata.obsm.keys())}")

# 上皮谱系筛选 -- EPITHELIAL_CLUSTERS 手动指定
if EPITHELIAL_CLUSTERS is not None:
    _epi_mask = adata.obs[_group_col].astype(str).isin(
        [str(c) for c in EPITHELIAL_CLUSTERS]
    )
    _n_before = adata.n_obs
    if not _epi_mask.any():
        raise ValueError(
            f"EPITHELIAL_CLUSTERS={EPITHELIAL_CLUSTERS} 不匹配任何细胞。"
            f"可用的 {_group_col} 值: "
            f"{sorted(adata.obs[_group_col].dropna().astype(str).unique())}"
        )
    adata = adata[_epi_mask].copy()
    print(
        f"上皮谱系筛选: {_n_before:,} -> {adata.n_obs:,} 细胞 "
        f"(保留 {_group_col}: {EPITHELIAL_CLUSTERS})"
    )
else:
    print(
        "EPITHELIAL_CLUSTERS=None，对所有细胞做拟时序。"
        "PI 可在完成细胞类型注释后指定上皮 cluster 重新分析。"
    )


In [ ]:
# === 上皮自动筛选（P2-6）===
# 在手动筛选之外，提供自动关键词检测筛选路径。EPITHELIAL_CLUSTERS 已指定时跳过。
if AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    _original_n = adata.n_obs
    if EPITHELIAL_LABEL_COL in adata.obs.columns:
        _labels = adata.obs[EPITHELIAL_LABEL_COL].astype(str).str.lower()
        _epi_mask = _labels.apply(lambda x: any(kw in x for kw in EPITHELIAL_KEYWORDS))

        if _epi_mask.sum() > 50:
            print(f"上皮自动筛选: {_epi_mask.sum():,}/{_original_n:,} cells 匹配上皮关键词")
            adata = adata[_epi_mask].copy()
            print(f"  筛选后: {adata.n_obs:,} cells")
            _top_labels = adata.obs[EPITHELIAL_LABEL_COL].value_counts().head(10)
            print(f"  包含标签: {_top_labels.to_dict()}")
        else:
            print(f"WARNING: 仅 {_epi_mask.sum()} cells 匹配上皮关键词（< 50），跳过筛选")
            print(f"  → 检查 EPITHELIAL_KEYWORDS 或 EPITHELIAL_LABEL_COL 是否正确")
            AUTO_SUBSET_EPITHELIAL = False
    else:
        print(f"WARNING: {EPITHELIAL_LABEL_COL} 列不存在，跳过上皮筛选")
        AUTO_SUBSET_EPITHELIAL = False
elif not AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    print("AUTO_SUBSET_EPITHELIAL=False 且 EPITHELIAL_CLUSTERS=None，使用全部细胞")
elif EPITHELIAL_CLUSTERS is not None:
    print(f"EPITHELIAL_CLUSTERS 已手动指定 ({EPITHELIAL_CLUSTERS})，跳过自动筛选")


In [ ]:
# === Root Cluster 识别 ===
# CellRank2 的 GPCCA 自动识别终末状态（terminal states），不需要手动指定 root。
# 但 CytoTRACEKernel 的分化方向受 CytoTRACE 得分影响，记录 root cluster
# 有助于后续横向比较各方法的 pseudotime 方向一致性。
# 策略（两级）：
#   1. 优先：从上游 h5ad 读取 adata.uns["root_cluster"]（10_pseudotime 已跑则存在）
#   2. Fallback：用干细胞 marker 均值最高的 cluster 作为 root

_top_cluster = None  # root cluster 标识
_present_markers = []  # 预初始化，确保所有分支都有定义

_root_from_upstream = False  # 追踪 root 来源，供输出 cell 使用
if USE_UPSTREAM_ROOT and "root_cluster" in adata.uns:
    _top_cluster = str(adata.uns["root_cluster"])
    _root_from_upstream = True
    print(f"从上游读取 root_cluster: {_top_cluster}")
    # 检查上游 root_cluster 是否在筛选后的 cluster 列表中
    _available_clusters = set(adata.obs[_group_col].astype(str).unique())
    if _top_cluster not in _available_clusters:
        print(f"WARNING: 上游 root_cluster '{_top_cluster}' 不在筛选后的 cluster 列表中")
        print(f"  可用 cluster: {sorted(_available_clusters)}")
        _top_cluster = None  # 触发 fallback
        _root_from_upstream = False  # 同步重置 provenance，确保 root_source 准确
else:
    if USE_UPSTREAM_ROOT:
        print("上游无 root_cluster（adata.uns['root_cluster'] 不存在），启用 stem marker fallback")
    else:
        print("USE_UPSTREAM_ROOT=False，强制使用 stem marker fallback")

# Fallback：干细胞 marker 均值最高的 cluster
if _top_cluster is None:
    _present_markers = [g for g in STEM_MARKERS if g in adata.var_names]
    _missing_markers = [g for g in STEM_MARKERS if g not in adata.var_names]
    print(f"干细胞 marker: {len(_present_markers)}/{len(STEM_MARKERS)} 个基因存在于数据集中")
    if _missing_markers:
        print(f"  缺失基因: {_missing_markers}")

    if len(_present_markers) == 0:
        raise ValueError(
            f"STEM_MARKERS 中所有基因都在数据集中不存在，无法做 root cluster fallback。"
            f"请：(1) 先跑 D04_pseudotime.ipynb 产生 root_cluster；"
            f"或 (2) 调整 STEM_MARKERS 列表"
        )

    # 计算每细胞的干细胞 marker 均值（优先 counts layer）
    if "counts" in adata.layers:
        _stem_expr = np.asarray(
            adata[:, _present_markers].layers["counts"].mean(axis=1)
        ).ravel()
    elif adata.raw is not None:
        _stem_expr = np.asarray(
            adata.raw[:, _present_markers].X.mean(axis=1)
        ).ravel()
    else:
        _stem_expr = np.asarray(
            adata[:, _present_markers].X.mean(axis=1)
        ).ravel()

    # 按簇聚合干细胞 marker 均值，选最高的簇为 root
    # NaN 检测：astype(str) 会把 NaN 无声转为字面 "nan"
    if adata.obs[_group_col].isna().any():
        print("WARNING: _group_col 含 NaN 值，将转为字面 'nan' 可能误导 root 识别")
    _groups = adata.obs[_group_col].astype(str)
    _stem_by_cluster = pd.Series(_stem_expr, index=adata.obs_names).groupby(_groups).mean()
    _stem_by_cluster = _stem_by_cluster.sort_values(ascending=False)
    _top_cluster = _stem_by_cluster.index[0]
    print(f"\n干细胞 marker 均值最高 cluster = {_top_cluster}")
    print(f"各簇干细胞 marker 均值（前 5）:")
    for _clu, _val in _stem_by_cluster.head(5).items():
        print(f"  cluster {_clu}: {_val:.6f}")

    if _top_cluster == "nan":
        print("WARNING: root cluster 退化为 'nan'，上游注释列可能全为空值，请检查数据质量")

# 写入 adata（供后续参考）
adata.uns["root_cluster"] = str(_top_cluster)
adata.obs["root_cluster"] = adata.obs[_group_col].astype(str)
print(f"\nroot_cluster 已写入: adata.uns['root_cluster'] = '{_top_cluster}'")
print(f"  adata.obs['root_cluster'] 覆盖 {adata.n_obs:,} 个细胞")


## CellRank2 Fate-Mapping 完整管线

**为什么用 CellRank2？** CellRank2 的核心优势在于其基于 Markov 转移矩阵的统计框架，
能同时完成三个层级的任务：
1. **分化潜能估计**：通过 CytoTRACEKernel 计算每个细胞的分化潜能评分（已在 `D04_pseudotime.ipynb` 中跑过）
2. **终末状态识别**：通过 GPCCA 的 Schur 分解自动识别谱系的终末状态（terminal states），
   无需人工指定谱系终点
3. **命运概率计算**：对每个细胞，计算其到达各终末状态的概率向量（fate probabilities）
   ——这是 Monocle3 和 DPT 无法提供的输出

**CellRank2 vs Monocle3 vs DPT**：三种方法各有侧重：
| 方法 | 伪时序输出 | 终末状态识别 | 命运概率 | 分支检测 |
|------|-----------|-------------|---------|---------|
| DPT | 连续 pseudotime | 手动指定 root | 无 | 无 |
| Monocle3 | 连续 pseudotime | 手动指定 root | 无 | 自动叶/分支节点 |
| CellRank2 | macrostates 级 | **自动**（GPCCA）| **有**（fate probs）| macrostates 间转移 |

**技术流程**：
1. 确保邻居图已构建（connectivities）
2. 选择 kernel 并构建 Markov 转移矩阵
3. GPCCA 分解转移矩阵，识别 macrostates 和终末状态
4. 计算 fate probabilities
5. 可视化 macrostates 和命运概率在 UMAP 上的分布

**优雅降级**：CellRank2 在小数据集或退化情况下可能报错（如 transition matrix 不满秩、
macrostates 无法收敛）。本 notebook 对所有 CellRank2 调用包 try/except，
捕获异常后 WARNING 并优雅跳过，确保 h5ad 仍能写出。


In [ ]:
# === 邻居图检查与构建 ===
# CellRank2 的转移矩阵依赖 connectivities（KNN 邻接图加权后的连接矩阵）。
# 如果上游 04/05 已建好 neighbors（如 scVI/scANVI 训练后），直接复用。
# 否则用最优 embedding 构建：X_scANVI > X_scVI > X_pca。

if RUN_CELLRANK2:
    if "neighbors" not in adata.uns:
        print("未检测到 neighbors -- 正在构建...")
        # 自动探测最优 embedding
        _use_rep = None
        for _key in adata.obsm.keys():
            if _key.lower().startswith("x_scanvi"):
                _use_rep = _key
                break
        if _use_rep is None:
            for _key in adata.obsm.keys():
                if _key.lower().startswith("x_scvi"):
                    _use_rep = _key
                    break
        if _use_rep is not None:
            print(f"  使用 embedding: {_use_rep}")
            sc.pp.neighbors(adata, use_rep=_use_rep, n_neighbors=CELLRANK_N_NEIGHBORS)
        elif "X_pca" in adata.obsm:
            print("  使用 X_pca 建 neighbors")
            sc.pp.neighbors(adata, n_neighbors=CELLRANK_N_NEIGHBORS)
        else:
            print("  无 embedding，先跑 PCA...")
            sc.tl.pca(adata, n_comps=50)
            sc.pp.neighbors(adata, n_neighbors=CELLRANK_N_NEIGHBORS)
        print("  neighbors 构建完成")
    else:
        print("复用上游已建的 neighbors")

    _has_conn = "connectivities" in adata.obsp
    print(f"connectivities 可用: {_has_conn}")
else:
    print("RUN_CELLRANK2=False，跳过 neighbors 检查")
    _has_conn = False


In [ ]:
# === CellRank2 Kernel 构建 + Markov 转移矩阵 ===
# 转移矩阵是 CellRank2 的核心——定义细胞间的转移概率。
# 转移方向取决于 kernel 选择：
# - CytoTRACEKernel：从低分化潜能 → 高分化潜能（默认，无需上游其他方法）
# - PseudotimeKernel：从 pseudotime 早期 → 晚期（需上游有 pseudotime 列）

_crk_success = False
_kernel = None

if RUN_CELLRANK2 and _has_conn:
    if CELLRANK_KERNEL == "cytotrace":
        print("=" * 60)
        print("CellRank2 Kernel: CytoTRACEKernel")
        print("=" * 60)
        try:
            from cellrank.kernels import CytoTRACEKernel

            ctk = CytoTRACEKernel(adata)
            # 显式指定 layer="X" 避免 cellrank 默认找 layer["imputed"]
            ctk.compute_cytotrace(layer="X")

            # 写回 CytoTRACE 得分（独立于 10_pseudotime 的 cytotrace_score）
            if hasattr(ctk, "cytotrace"):
                adata.obs["cellrank2_cytotrace"] = (
                    np.asarray(ctk.cytotrace).ravel().astype(np.float32)
                )
                _ct = adata.obs["cellrank2_cytotrace"]
                print(f"CytoTRACE 得分: mean={_ct.mean():.4f}, "
                      f"range=[{_ct.min():.4f}, {_ct.max():.4f}]")
            elif "ct_score" in adata.obs.columns:
                adata.obs["cellrank2_cytotrace"] = adata.obs["ct_score"].astype(np.float32)
                print("CytoTRACE 得分已从 ct_score 读取")
            else:
                _candidates = [c for c in adata.obs.columns
                               if "cyto" in c.lower() or "trace" in c.lower()]
                if _candidates:
                    adata.obs["cellrank2_cytotrace"] = pd.to_numeric(
                        adata.obs[_candidates[0]], errors="coerce"
                    ).astype(np.float32)
                    print(f"CytoTRACE 得分已从 {_candidates[0]} 读取")

            # 构建 Markov 转移矩阵
            ctk.compute_transition_matrix()
            _kernel = ctk
            _crk_success = True
            print("CytoTRACEKernel 转移矩阵构建成功")
        except Exception as _e:
            print(f"WARNING: CytoTRACEKernel 失败: {_e}")
            import traceback
            traceback.print_exc()
            print("  CellRank2 fate-mapping 将跳过，h5ad 仍会写出")

    elif CELLRANK_KERNEL == "pseudotime":
        print("=" * 60)
        print("CellRank2 Kernel: PseudotimeKernel")
        print("=" * 60)
        try:
            from cellrank.kernels import PseudotimeKernel

            # 确定 pseudotime 列
            _pt_key = PSEUDOTIME_KEY
            if _pt_key is None:
                for _cand in ["pseudotime_monocle3_v1", "dpt_pseudotime", "pseudotime"]:
                    if _cand in adata.obs.columns:
                        _n_valid = adata.obs[_cand].notna().sum()
                        if _n_valid > 50:
                            _pt_key = _cand
                            print(f"自动探测 pseudotime 列: {_pt_key} ({_n_valid} valid cells)")
                            break

            if _pt_key is None or _pt_key not in adata.obs.columns:
                raise RuntimeError(
                    f"PSEUDOTIME_KEY={PSEUDOTIME_KEY}，"
                    f"且自动探测未找到可用 pseudotime 列。"
                    f"请先跑 10b (Monocle3) 或 10_pseudotime 产生 pseudotime"
                )

            if adata.obs[_pt_key].notna().sum() < 50:
                raise RuntimeError(
                    f"pseudotime 列 '{_pt_key}' 有效值不足 50 个"
                )

            pk = PseudotimeKernel(adata, time_key=_pt_key)
            pk.compute_transition_matrix()
            _kernel = pk
            _crk_success = True
            print(f"PseudotimeKernel 转移矩阵构建成功 (time_key={_pt_key})")
        except Exception as _e:
            print(f"WARNING: PseudotimeKernel 失败: {_e}")
            import traceback
            traceback.print_exc()
            print("  CellRank2 fate-mapping 将跳过，h5ad 仍会写出")

    else:
        print(f"WARNING: 未知 CELLRANK_KERNEL='{CELLRANK_KERNEL}'，"
              f"支持: 'cytotrace' | 'pseudotime'")
else:
    _reasons = []
    if not RUN_CELLRANK2:
        _reasons.append("RUN_CELLRANK2=False")
    if not _has_conn:
        _reasons.append("connectivities 不存在")
    print(f"跳过 kernel 构建（{'；'.join(_reasons)}）")


In [ ]:
# === GPCCA Estimator：终末状态识别 + 命运概率 ===
# GPCCA（Generalized Perron Cluster Cluster Analysis）基于 Markov 转移矩阵
# 的 Schur 分解，在特征向量空间中识别粗粒化的"宏观状态"（macrostates），
# 自动推断谱系的终末状态（terminal states），并计算每细胞到达各终末状态的概率。
#
# 步骤：
#   1. compute_schur()    -- Schur 分解转移矩阵
#   2. compute_macrostates()  -- 从 Schur 向量聚类识别 macrostates
#   3. predict_terminal_states()  -- 从 macrostates 自动识别终末状态
#   4. compute_fate_probabilities() -- 吸收 Markov 链求 fate probabilities

_gpcca_success = False
_terminal_states_list = []
_actual_n_states = CELLRANK_N_STATES

if RUN_CELLRANK2 and _crk_success:
    print("=" * 60)
    print("CellRank2 GPCCA Estimator")
    print("=" * 60)
    try:
        from cellrank.estimators import GPCCA

        g = GPCCA(_kernel)

        # Step 1: Schur 分解
        # n_components 应 >= 期望的 macrostates 数量
        _n_comp = max(CELLRANK_N_STATES if CELLRANK_N_STATES is not None else 10, 10)
        g.compute_schur(n_components=_n_comp)
        print(f"Schur 分解完成 (n_components={_n_comp})")

        # Step 2: 识别 macrostates
        # cluster_key 约束 macrostates 为细胞类型簇的并集（提高可解释性）
        # 如果 CELL_TYPE_COL 全 NaN 或无效，则不传 cluster_key
        _cluster_key = None
        if CELL_TYPE_COL in adata.obs.columns:
            _valid_ct = adata.obs[CELL_TYPE_COL].dropna()
            if len(_valid_ct) > 0 and _valid_ct.nunique() >= 2:
                _cluster_key = CELL_TYPE_COL
                print(f"使用 cluster_key: {CELL_TYPE_COL}")
            else:
                print(
                    f"WARNING: {CELL_TYPE_COL} 无效（全 NaN 或单一值），"
                    f"GPCCA 将不使用 cluster_key 约束"
                )
        else:
            print(f"WARNING: {CELL_TYPE_COL} 列不存在，GPCCA 将不使用 cluster_key 约束")

        g.compute_macrostates(
            n_states=CELLRANK_N_STATES,
            cluster_key=_cluster_key,
        )

        # 提取实际 macrostates 数量
        if hasattr(g, "macrostates") and g.macrostates is not None:
            try:
                _actual_n_states = len(g.macrostates.cat.categories)
            except (AttributeError, TypeError):
                _actual_n_states = g.macrostates.nunique()
            print(f"Macrostates 识别完成: {_actual_n_states} 个")
        else:
            print("Macrostates 识别完成（无法提取数量）")

        # Step 3: 预测终末状态
        g.predict_terminal_states()
        if hasattr(g, "terminal_states") and g.terminal_states is not None:
            try:
                _terminal_states_list = sorted(
                    g.terminal_states.cat.categories.tolist()
                )
            except (AttributeError, TypeError):
                _terminal_states_list = sorted(
                    g.terminal_states.dropna().unique().tolist()
                )
            print(f"终末状态: {_terminal_states_list}")
        else:
            print("WARNING: 未检测到 terminal_states 属性")

        # Step 4: 命运概率（核心产出）
        g.compute_fate_probabilities()
        print("Fate probabilities 计算完成")

        _gpcca_success = True

        # 检查 fate_probabilities 属性
        if hasattr(g, "fate_probabilities") and g.fate_probabilities is not None:
            _fp = g.fate_probabilities
            if hasattr(_fp, "shape"):
                print(f"  fate_probabilities shape: {_fp.shape}")
            if hasattr(_fp, "names"):
                print(f"  lineage names: {list(_fp.names)}")

    except Exception as _e:
        print(f"WARNING: GPCCA 失败: {_e}")
        import traceback
        traceback.print_exc()
        print("  CellRank2 fate-mapping 将跳过，h5ad 仍会写出")
else:
    _reasons = []
    if not RUN_CELLRANK2:
        _reasons.append("RUN_CELLRANK2=False")
    if not _crk_success:
        _reasons.append("kernel 构建失败")
    print(f"跳过 GPCCA（{'；'.join(_reasons)}）")


In [ ]:
# === CellRank2 结果写回 adata ===
# 将 GPCCA 产出的 macrostates / terminal_states / fate_probabilities
# 显式写入 adata.obs 和 adata.obsm，确保 checkpoint h5ad 自包含。

if RUN_CELLRANK2 and _crk_success and _gpcca_success:
    _written = []

    # Macrostate 分配
    try:
        if hasattr(g, "macrostates") and g.macrostates is not None:
            _ms = g.macrostates
            if hasattr(_ms, "values"):
                _ms_vals = _ms.values
            elif hasattr(_ms, "to_numpy"):
                _ms_vals = _ms.to_numpy()
            else:
                _ms_vals = np.asarray(_ms)
            adata.obs["cellrank2_macrostate"] = pd.Series(
                _ms_vals, index=adata.obs_names
            ).astype(str).values
            _n = adata.obs["cellrank2_macrostate"].nunique()
            print(f"cellrank2_macrostate 已写入 ({_n} 个状态)")
            _written.append("cellrank2_macrostate")
    except Exception as _e:
        print(f"WARNING: 写 macrostate 失败: {_e}")

    # Terminal state 标签
    try:
        if hasattr(g, "terminal_states") and g.terminal_states is not None:
            _ts = g.terminal_states
            if hasattr(_ts, "values"):
                _ts_vals = _ts.values
            elif hasattr(_ts, "to_numpy"):
                _ts_vals = _ts.to_numpy()
            else:
                _ts_vals = np.asarray(_ts)
            adata.obs["cellrank2_terminal_state"] = pd.Series(
                _ts_vals, index=adata.obs_names
            ).astype(str).values
            _n = adata.obs["cellrank2_terminal_state"].nunique()
            print(f"cellrank2_terminal_state 已写入 ({_n} 个终末状态)")
            _written.append("cellrank2_terminal_state")
    except Exception as _e:
        print(f"WARNING: 写 terminal_state 失败: {_e}")

    # Fate probabilities 矩阵
    try:
        if hasattr(g, "fate_probabilities") and g.fate_probabilities is not None:
            _fp = g.fate_probabilities
            # 提取数值矩阵
            if hasattr(_fp, "X"):
                _fp_mat = _fp.X
            elif hasattr(_fp, "values"):
                _fp_mat = _fp.values
            else:
                _fp_mat = np.asarray(_fp)

            if sp.issparse(_fp_mat):
                _fp_mat = _fp_mat.toarray()
            adata.obsm["cellrank2_fate_probabilities"] = _fp_mat.astype(np.float32)

            # 记录列名（lineage/终末状态名）
            if hasattr(_fp, "names"):
                adata.uns["cellrank2_fate_probabilities_names"] = list(_fp.names)
            elif hasattr(_fp, "columns"):
                adata.uns["cellrank2_fate_probabilities_names"] = list(_fp.columns)
            else:
                adata.uns["cellrank2_fate_probabilities_names"] = [
                    str(s) for s in _terminal_states_list
                ]
            print(
                f"cellrank2_fate_probabilities 已写入 obsm "
                f"(shape={_fp_mat.shape})"
            )
            _written.append("cellrank2_fate_probabilities")
    except Exception as _e:
        print(f"WARNING: 写 fate_probabilities 失败: {_e}")

    # GPCCA 伪时序（如果 GPCCA 计算了方向性 pseudotime）
    try:
        if hasattr(g, "pseudotime") and g.pseudotime is not None:
            _pt = g.pseudotime
            if hasattr(_pt, "values"):
                _pt_vals = _pt.values
            elif hasattr(_pt, "to_numpy"):
                _pt_vals = _pt.to_numpy()
            else:
                _pt_vals = np.asarray(_pt)
            adata.obs["cellrank2_pseudotime"] = pd.Series(
                _pt_vals.ravel(), index=adata.obs_names
            ).astype(np.float32).values
            print("cellrank2_pseudotime 已写入")
            _written.append("cellrank2_pseudotime")
    except Exception as _e:
        pass  # GPCCA 不一定产出 pseudotime，非必需

    # ---- CellRank 内部 obsm key 清理 ----
    # GPCCA 在 adata.obsm 中存储 Lineage 对象（如 macrostates_fwd_memberships），
    # 这些对象的 HDF5 序列化可能与已安装的 anndata 版本不兼容。
    # 遍历所有 obsm key，将 Lineage 对象转为纯 numpy array，无法转换则删除。
    _cleaned = 0
    for _key in list(adata.obsm.keys()):
        _val = adata.obsm[_key]
        _type = type(_val)
        _type_qualname = f"{getattr(_type, '__module__', '')}.{getattr(_type, '__qualname__', '')}"
        if "cellrank" in _type_qualname.lower() or "lineage" in _type_qualname.lower():
            try:
                if hasattr(_val, "X"):
                    adata.obsm[_key] = np.asarray(_val.X).astype(np.float32)
                else:
                    adata.obsm[_key] = np.asarray(_val).astype(np.float32)
                _cleaned += 1
                print(f"  已转换 cellrank 内部 key: {_key} -> numpy array")
            except Exception:
                del adata.obsm[_key]
                _cleaned += 1
                print(f"  已删除无法转换的 cellrank 内部 key: {_key}")
    if _cleaned > 0:
        print(f"CellRank 内部 key 清理完成: {_cleaned} 个")

    # ---- obsp Lineage 对象清理 ----
    # compute_transition_matrix() 可能向 obsp 写入 cellrank 内部对象（如 "T_fwd"）。
    # 虽然通常是 scipy 稀疏矩阵，但为稳健起见做防御性清理。
    _obsp_cleaned = 0
    for _key in list(adata.obsp.keys()):
        _val = adata.obsp[_key]
        _type = type(_val)
        _type_qualname = f"{getattr(_type, '__module__', '')}.{getattr(_type, '__qualname__', '')}"
        if "cellrank" in _type_qualname.lower() or "lineage" in _type_qualname.lower():
            try:
                if hasattr(_val, "X"):
                    adata.obsp[_key] = np.asarray(_val.X).astype(np.float32)
                else:
                    adata.obsp[_key] = np.asarray(_val).astype(np.float32)
                _obsp_cleaned += 1
                print(f"  已转换 cellrank 内部 obsp key: {_key} -> numpy array")
            except Exception:
                del adata.obsp[_key]
                _obsp_cleaned += 1
                print(f"  已删除无法转换的 cellrank 内部 obsp key: {_key}")
    if _obsp_cleaned > 0:
        print(f"CellRank 内部 obsp key 清理完成: {_obsp_cleaned} 个")

    # ---- Uns 中的 Lineage 对象清理 ----
    for _key in list(adata.uns.keys()):
        try:
            _val = adata.uns[_key]
            _type = type(_val)
            _type_qualname = f"{getattr(_type, '__module__', '')}.{getattr(_type, '__qualname__', '')}"
            if "cellrank" in _type_qualname.lower() or "lineage" in _type_qualname.lower():
                del adata.uns[_key]
                print(f"  已删除 cellrank 内部 uns key: {_key}")
        except Exception:
            pass

    print(f"CellRank2 结果写回完成: {_written}")
else:
    _reasons = []
    if not RUN_CELLRANK2:
        _reasons.append("RUN_CELLRANK2=False")
    if not _crk_success:
        _reasons.append("kernel 失败")
    if not _gpcca_success:
        _reasons.append("GPCCA 失败")
    print(f"CellRank2 结果未写入（{'；'.join(_reasons)}）")

In [ ]:
# === CellRank2 可视化 ===
# 画 macrostates 在 UMAP 上的分布 + fate probabilities。
# 优先使用 cellrank 内置 plotting（如 g.plot_macrostates），
# 不可用时 fallback 到手动 matplotlib。

if RUN_CELLRANK2 and _crk_success and _gpcca_success:
    _fig_dir = "results/figures"

    # ----- 1. Macrostates on UMAP -----
    _macro_path = os.path.join(_fig_dir, "10c_macrostates_umap.png")
    try:
        # 尝试 cellrank 内置 plotting
        if hasattr(g, "plot_macrostates"):
            _ax = g.plot_macrostates(title="CellRank2 Macrostates", show=False)
            if _ax is not None:
                _fig = _ax.figure if hasattr(_ax, "figure") else plt.gcf()
                _fig.savefig(_macro_path, dpi=200, bbox_inches="tight")
                plt.close(_fig)
                print(f"Macrostates UMAP 已保存: {_macro_path}")
            else:
                raise RuntimeError("plot_macrostates 返回 None")
        else:
            raise AttributeError("无 plot_macrostates 方法")
    except Exception as _e:
        print(f"cellrank 内置 plotting 不可用 ({_e})，fallback 到手动绘图")
        try:
            if "X_umap" in adata.obsm and "cellrank2_macrostate" in adata.obs.columns:
                fig, ax = plt.subplots(figsize=(8, 7))
                _coords = adata.obsm["X_umap"]
                _cats = adata.obs["cellrank2_macrostate"].astype(str)
                for _cat in sorted(_cats.unique()):
                    _mask = _cats == _cat
                    ax.scatter(
                        _coords[_mask, 0], _coords[_mask, 1],
                        s=3, alpha=0.7, label=_cat, rasterized=True,
                    )
                ax.set_title("CellRank2 Macrostates (manual fallback)")
                ax.set_xlabel("UMAP 1")
                ax.set_ylabel("UMAP 2")
                ax.legend(loc="upper right", frameon=False, markerscale=3,
                          title="Macrostate", fontsize=7)
                fig.savefig(_macro_path, dpi=200, bbox_inches="tight")
                plt.close(fig)
                print(f"Macrostates UMAP (manual) 已保存: {_macro_path}")
        except Exception as _e2:
            print(f"WARNING: 手动 macrostates 绘图也失败: {_e2}")

    # ----- 2. Fate probabilities on UMAP -----
    _fate_path = os.path.join(_fig_dir, "10c_fate_probabilities_umap.png")
    try:
        # 尝试 cellrank 内置 plotting
        if hasattr(g, "plot_fate_probabilities"):
            _ax = g.plot_fate_probabilities(
                basis="umap", title="CellRank2 Fate Probabilities", show=False,
            )
            if _ax is not None:
                _figs = _ax if isinstance(_ax, (list, tuple)) else [_ax]
                for _i, _a in enumerate(_figs):
                    _fig = _a.figure if hasattr(_a, "figure") else plt.gcf()
                    if len(_figs) == 1:
                        _fig.savefig(_fate_path, dpi=200, bbox_inches="tight")
                    else:
                        _fp = _fate_path.replace(".png", f"_{_i}.png")
                        _fig.savefig(_fp, dpi=200, bbox_inches="tight")
                    plt.close(_fig)
                print(f"Fate probabilities UMAP 已保存: {_fate_path}")
            else:
                raise RuntimeError("plot_fate_probabilities 返回 None")
        else:
            raise AttributeError("无 plot_fate_probabilities 方法")
    except Exception as _e:
        print(f"cellrank 内置 fate plotting 不可用 ({_e})，fallback 到手动绘图")
        try:
            if ("X_umap" in adata.obsm
                    and "cellrank2_fate_probabilities" in adata.obsm):
                _coords = adata.obsm["X_umap"]
                _fp_mat = adata.obsm["cellrank2_fate_probabilities"]
                _names = adata.uns.get(
                    "cellrank2_fate_probabilities_names",
                    [f"lineage_{i}" for i in range(_fp_mat.shape[1])]
                )

                _n_lineages = _fp_mat.shape[1]
                _ncols = min(3, _n_lineages)
                _nrows = (_n_lineages + _ncols - 1) // _ncols
                fig, axes = plt.subplots(
                    _nrows, _ncols,
                    figsize=(5 * _ncols, 4.5 * _nrows),
                    squeeze=False,
                )

                for _i in range(_n_lineages):
                    ax = axes[_i // _ncols, _i % _ncols]
                    _sc = ax.scatter(
                        _coords[:, 0], _coords[:, 1],
                        c=_fp_mat[:, _i], cmap="viridis",
                        s=3, alpha=0.7, rasterized=True,
                    )
                    plt.colorbar(_sc, ax=ax, label="Fate Probability")
                    ax.set_title(_names[_i], fontsize=11)
                    ax.set_xlabel("UMAP 1")
                    ax.set_ylabel("UMAP 2")

                # 隐藏多余子图
                for _i in range(_n_lineages, _nrows * _ncols):
                    axes[_i // _ncols, _i % _ncols].set_visible(False)

                plt.suptitle("CellRank2 Fate Probabilities", fontsize=13,
                             fontweight="bold")
                plt.tight_layout()
                fig.savefig(_fate_path, dpi=200, bbox_inches="tight")
                plt.close(fig)
                print(f"Fate probabilities UMAP (manual) 已保存: {_fate_path}")
        except Exception as _e2:
            print(f"WARNING: 手动 fate probability 绘图也失败: {_e2}")

else:
    print("CellRank2 未成功运行，跳过可视化")


In [ ]:
# 内存自检 -- 确保 adata.X 稀疏性/精度在 CellRank2 流程中未被破坏。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增 obs 列
_new_cols = ["cellrank2_cytotrace", "cellrank2_macrostate", "cellrank2_terminal_state"]
for _c in _new_cols:
    if _c in adata.obs.columns:
        _v = adata.obs[_c]
        print(f"  {_c}: non-NA={_v.notna().sum()}/{len(_v)}")
    else:
        print(f"  {_c}: 未写入")

# 检查 obsm 新增 key
if "cellrank2_fate_probabilities" in adata.obsm:
    _fps = adata.obsm["cellrank2_fate_probabilities"].shape
    print(f"  cellrank2_fate_probabilities: shape={_fps}")
else:
    print("  cellrank2_fate_probabilities: 未写入")


In [ ]:
# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "10c_pseudotime_cellrank2"
adata.uns["version"] = OUTPUT_VERSION
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# 若 GPCCA 未成功运行，_actual_n_states 可能仍为 None；
# 此时若已知终末状态列表，用其长度做最佳估计
if _actual_n_states is None and _terminal_states_list:
    _actual_n_states = len(_terminal_states_list)

# CellRank2 方法细节——嵌套 dict 记录参数（与其他 stage 的嵌套 dict 形态一致）
_cellrank_uns = {
    "method": "cellrank2",
    "kernel": CELLRANK_KERNEL,
    "n_neighbors": CELLRANK_N_NEIGHBORS,
    "n_states": _actual_n_states,
    "terminal_states": _terminal_states_list,
    "run_cellrank2": RUN_CELLRANK2,
    "kernel_success": _crk_success if "_crk_success" in dir() else False,
    "gpcca_success": _gpcca_success if "_gpcca_success" in dir() else False,
    "root_cluster": str(adata.uns.get("root_cluster", "")),
    "root_source": "upstream" if ("_root_from_upstream" in dir() and _root_from_upstream) else "stem_marker_fallback",
    "epithelial_clusters": EPITHELIAL_CLUSTERS,
    "auto_subset_epithelial": AUTO_SUBSET_EPITHELIAL,
    "group_col": _group_col,
    "stem_markers_used": _present_markers,
    "n_cells": adata.n_obs,
    "pseudotime_key": PSEUDOTIME_KEY,
}
adata.uns[f"10c_pseudotime_cellrank2_{OUTPUT_VERSION}"] = _cellrank_uns
print(
    f"追踪字段已写入: stage={adata.uns['stage']}  "
    f"version={adata.uns['version']}  "
    f"status={adata.uns['status']}"
)

In [ ]:
# 写出 checkpoint（带序列化兜底）。
# CellRank2 可能在 obsm/obsp/uns 中残留非标准类型对象，
# 导致 adata.write 的 HDF5 序列化失败。先尝试正常写出；
# 若失败，做一次深度二次清理后重试一次。

def _is_safe_type(_val):
    """判断是否为 HDF5 可安全序列化的类型。"""
    if isinstance(_val, np.ndarray):
        return True
    if sp.issparse(_val):
        return True
    if isinstance(_val, (str, int, float, bool, list, dict, tuple, type(None),
                         np.integer, np.floating)):
        return True
    return False

_wrote = False
try:
    adata.write_h5ad(OUTPUT_PATH, compression="lzf")
    _wrote = True
    print(f"已写出 {OUTPUT_PATH}")
except Exception as _write_err:
    print(f"WARNING: 首次 write_h5ad 失败: {_write_err}")
    print("  执行深度二次清理（移除所有非标准类型对象）...")

    # obsm 二次清理
    for _key in list(adata.obsm.keys()):
        _val = adata.obsm[_key]
        if _is_safe_type(_val):
            continue
        try:
            if hasattr(_val, "X"):
                adata.obsm[_key] = np.asarray(_val.X).astype(np.float32)
                print(f"  深度清理 obsm['{_key}']: 转 numpy (via .X)")
            elif hasattr(_val, "toarray"):
                adata.obsm[_key] = np.asarray(_val.toarray()).astype(np.float32)
                print(f"  深度清理 obsm['{_key}']: 转 numpy (via .toarray)")
            elif hasattr(_val, "__array__"):
                adata.obsm[_key] = np.asarray(_val).astype(np.float32)
                print(f"  深度清理 obsm['{_key}']: 转 numpy (via __array__)")
            else:
                del adata.obsm[_key]
                print(f"  深度清理 obsm['{_key}']: 已删除 (type={type(_val).__name__})")
        except Exception as _e2:
            del adata.obsm[_key]
            print(f"  深度清理 obsm['{_key}']: 已删除 (转换失败: {_e2})")

    # obsp 二次清理
    for _key in list(adata.obsp.keys()):
        _val = adata.obsp[_key]
        if _is_safe_type(_val):
            continue
        try:
            if hasattr(_val, "X"):
                adata.obsp[_key] = np.asarray(_val.X).astype(np.float32)
                print(f"  深度清理 obsp['{_key}']: 转 numpy (via .X)")
            elif hasattr(_val, "toarray"):
                adata.obsp[_key] = np.asarray(_val.toarray()).astype(np.float32)
                print(f"  深度清理 obsp['{_key}']: 转 numpy (via .toarray)")
            elif hasattr(_val, "__array__"):
                adata.obsp[_key] = np.asarray(_val).astype(np.float32)
                print(f"  深度清理 obsp['{_key}']: 转 numpy (via __array__)")
            else:
                del adata.obsp[_key]
                print(f"  深度清理 obsp['{_key}']: 已删除 (type={type(_val).__name__})")
        except Exception as _e2:
            del adata.obsp[_key]
            print(f"  深度清理 obsp['{_key}']: 已删除 (转换失败: {_e2})")

    # uns 二次清理
    for _key in list(adata.uns.keys()):
        _val = adata.uns[_key]
        if _is_safe_type(_val):
            continue
        try:
            _type_name = f"{getattr(type(_val), '__module__', '')}.{getattr(type(_val), '__qualname__', type(_val).__name__)}"
            if "cellrank" in _type_name.lower() or "lineage" in _type_name.lower():
                del adata.uns[_key]
                print(f"  深度清理 uns['{_key}']: 已删除 (type={_type_name})")
        except Exception:
            pass

    # 重试写出
    try:
        adata.write_h5ad(OUTPUT_PATH, compression="lzf")
        _wrote = True
        print(f"深度清理后写出成功: {OUTPUT_PATH}")
    except Exception as _retry_err:
        print(f"ERROR: 深度清理后 write_h5ad 仍失败: {_retry_err}")
        # 打印残留非标准类型的详细信息
        for _scope, _container in [("obsm", adata.obsm), ("obsp", adata.obsp), ("uns", adata.uns)]:
            for _key in list(_container.keys()):
                _val = _container[_key]
                if not _is_safe_type(_val):
                    _type_info = f"{getattr(type(_val), '__module__', '')}.{getattr(type(_val), '__qualname__', type(_val).__name__)}"
                    print(f"  残留非标准类型: {_scope}['{_key}'] -> {_type_info}")
        raise

assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")
